In [ ]:
# # To fix version clashes that prevent required libraries of working together, i need to pin versions:
# !pip install \                            
#   "numpy==1.26.4" \
#   "pandas==2.1.4" \
#   "spacy==3.7.5" \
#   "spacy-transformers==1.3.5" \
#   "torch==2.2.2"

# # Test that install works:
# import numpy, pandas, spacy
# print("NumPy:", numpy.__version__)
# print("pandas:", pandas.__version__)
# nlp = spacy.load("en_core_web_trf")
# print(nlp.pipe_names)


In [1]:
import pandas as pd
import spacy
# !python3 -m spacy download en_core_web_trf

import json
from openai import OpenAI
from tqdm import tqdm


Ingest sentence CSV to source target sentences

In [54]:
sent_sg = pd.read_csv('/Users/rdubnic2/Desktop/htdl-ocr-project/sent_search/difficult_matches_matched5_status.csv', index_col=0)
sent_sg = sent_sg.rename(columns={
    "hid": "htid",
    "gsent": "target_gsent",
    "hsent": "target_hsent",
    "matched_honed_lev": "matched_hsent",
    "matched_honed_lev_score": "matched_hsent_lev_score"
    })

print(sent_sg.shape)
sent_sg.head()

(62322, 19)


,csv_source,domain,gid,htid,pubdate,cidx,sidx,cer,wer,target_gsent,target_hsent,hsent_norm,matched_sentence,matched_sent_idx,score_containment,matched_honed_original,matched_hsent,matched_hsent_lev_score,match_status
idx,,,,,,,,,,,,,,,,,,,
0,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,2,0.093750,0.200000,""" Scarcely so much , mamma ; indeed , it wants...",""" Scarcely so much , mamma ; indeed it wants a...",scarcely so much mamma indeed it wants a quart...,""" Scarcely so much, mamma ; indeed it wants a ...",154.0,1.000000,""" Scarcely so much, mamma ; indeed it wants a ...",""" Scarcely so much, mamma ; indeed it wants a ...",0.953846,ok
1,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,3,0.080000,0.285714,The coach does not arrive till half past eigh...,the coach does not an ive till half past eig...,the coach does not an ive till half past eight...,""" Scarcely so much, mamma ; indeed it wants a ...",154.0,1.000000,the coach does not an-ive till half-past eight...,the coach does not an-ive till half-past eight...,0.902913,ok
2,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,5,0.032000,0.120000,"this waiting , it destroys me , "" rejoined the...","this waiting , it destroys me ; "" rejoined the...",this waiting it destroys me rejoined the first...,"this waiting, it destroys me ;"" rejoined the f...",156.0,1.000000,"this waiting, it destroys me ;"" rejoined the f...","this waiting, it destroys me ;"" rejoined the f...",0.960630,ok
3,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,6,0.036585,0.105263,""" How you can contrive to sit there , drawing ...","' how you can contrive to sit there , drawing...",how you can contrive to sit there drawing so q...,"this waiting, it destroys me ;"" rejoined the f...",156.0,1.000000,NaN,"'• how you can contrive to sit there, drawing ...",0.927711,ok
4,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,8,0.054054,0.217391,"Why did you not tell me so before ? "" returned...","why did you not teU me so before 1 "" returned ...",why did you not teu me so before 1 returned ro...,""" Does it annoy you, dear mamma 1 — why did yo...",157.0,0.947368,"why did you not teU \n me so before 1"" returne...","why did you not teU \n me so before 1"" returne...",0.918919,ok


In [55]:
# Slice Sarah's data to only keep matched sentences
print(sent_sg.shape)

# drop NaNs
sent_sg = sent_sg[sent_sg["htid"] != "hvd.hn6nvw"]
print(sent_sg.shape)
sent_sg = sent_sg[sent_sg["match_status"] != "no_match"]
print(sent_sg.shape)

sent_sg.head()

(62322, 19)
(62099, 19)
(59760, 19)


,csv_source,domain,gid,htid,pubdate,cidx,sidx,cer,wer,target_gsent,target_hsent,hsent_norm,matched_sentence,matched_sent_idx,score_containment,matched_honed_original,matched_hsent,matched_hsent_lev_score,match_status
idx,,,,,,,,,,,,,,,,,,,
0,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,2,0.093750,0.200000,""" Scarcely so much , mamma ; indeed , it wants...",""" Scarcely so much , mamma ; indeed it wants a...",scarcely so much mamma indeed it wants a quart...,""" Scarcely so much, mamma ; indeed it wants a ...",154.0,1.000000,""" Scarcely so much, mamma ; indeed it wants a ...",""" Scarcely so much, mamma ; indeed it wants a ...",0.953846,ok
1,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,3,0.080000,0.285714,The coach does not arrive till half past eigh...,the coach does not an ive till half past eig...,the coach does not an ive till half past eight...,""" Scarcely so much, mamma ; indeed it wants a ...",154.0,1.000000,the coach does not an-ive till half-past eight...,the coach does not an-ive till half-past eight...,0.902913,ok
2,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,5,0.032000,0.120000,"this waiting , it destroys me , "" rejoined the...","this waiting , it destroys me ; "" rejoined the...",this waiting it destroys me rejoined the first...,"this waiting, it destroys me ;"" rejoined the f...",156.0,1.000000,"this waiting, it destroys me ;"" rejoined the f...","this waiting, it destroys me ;"" rejoined the f...",0.960630,ok
3,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,6,0.036585,0.105263,""" How you can contrive to sit there , drawing ...","' how you can contrive to sit there , drawing...",how you can contrive to sit there drawing so q...,"this waiting, it destroys me ;"" rejoined the f...",156.0,1.000000,NaN,"'• how you can contrive to sit there, drawing ...",0.927711,ok
4,highpairs,Fiction,56600,uc2.ark+=13960=t7dr2vk0r,1870,0,8,0.054054,0.217391,"Why did you not tell me so before ? "" returned...","why did you not teU me so before 1 "" returned ...",why did you not teu me so before 1 returned ro...,""" Does it annoy you, dear mamma 1 — why did yo...",157.0,0.947368,"why did you not teU \n me so before 1"" returne...","why did you not teU \n me so before 1"" returne...",0.918919,ok


In [56]:
mask = (sent_sg["match_status"] == "ok") & (sent_sg["matched_hsent"].isna())

count_missing = mask.sum()
print(count_missing)

mask_bad = (
    (sent_sg["match_status"] == "ok") &
    (sent_sg["matched_hsent"].isna() | (sent_sg["matched_hsent"].astype(str).str.strip() == ""))
)

sent_sg = sent_sg[~mask_bad]

print(sent_sg.shape)

2835
(56925, 19)


In [57]:
# Load my found sentences
sent_rd = pd.read_csv('/Users/rdubnic2/Desktop/htdl-ocr-project/sent_search/sent_matches_deduped_final_102495.csv')
print(sent_rd.shape)

# concatenate the two sentence DFs together for final found sentence DF:
sent_df = pd.concat([sent_rd, sent_sg], ignore_index=True, sort=False)
print(sent_df.shape)
print(f"Target: 159420")

sent_df.head()

(102495, 15)
(159420, 22)
Target: 159420


,htid,target_hsent,target_gsent,gid,source_row_index,matched_hsent,source_df,csv_source,domain,pubdate,...,cer,wer,lev_similarity,hsent_norm,matched_sentence,matched_sent_idx,score_containment,matched_honed_original,matched_hsent_lev_score,match_status
0,uc2.ark+=13960=t7dr2vk0r,Both she and her mother were clad in the deepe...,Both she and her mother were clad in the deepe...,56600,6.0,Both she \n and her mother were clad in the d...,ht_matches_4,highpairs,Fiction,1870,...,0.014085,0.037037,0.940397,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,uc2.ark+=13960=t7dr2vk0r,"At length , suddenly breaking oiF , she exclai...","At length , suddenly breaking off , she exclai...",56600,7.0,"At length , suddenly breaking oiF , she exclai...",ht_matches_4,highpairs,Fiction,1870,...,0.022472,0.052632,0.946809,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,uc2.ark+=13960=t7dr2vk0r,"what am I singing f and , bmying her face in h...","what am I singing ? "" and , burying her face i...",56600,8.0,"what am I singing f and , \n \n bmying her fa...",ht_matches_4,highpairs,Fiction,1870,...,0.062500,0.156250,0.898734,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,uc2.ark+=13960=t7dr2vk0r,"Besides , you would be miserable , managing na...","Besides , you would be miserable , managing na...",56600,10.0,"Besides , you would be miserable , managing \n...",ht_matches_4,highpairs,Fiction,1870,...,0.054348,0.138889,0.915423,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,uc2.ark+=13960=t7dr2vk0r,""" And do you think , mamma , that I could be c...",""" And do you think , mamma , that I could be c...",56600,11.0,""" And do you think , mamma , that I could be c...",ht_matches_4,highpairs,Fiction,1870,...,0.032258,0.075000,0.949495,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Below code took about 100 minutes for full 102k-row run

In [9]:
# NER with spaCy

nlp = spacy.load("en_core_web_trf")

# df = pd.read_csv("sent_matches_nodupes_102615_final.csv")
df = sent_df

TARGET_H = "target_hsent"
TARGET_G = "target_gsent"
MATCHED_H = "matched_hsent"

def extract_ents_spacy_batch(texts, batch_size=50):
    """
    texts: iterable of strings (Series or list)
    batch_size: increase if you have lots of RAM (64 or 128 can work)
    """
    results = []
    for doc in nlp.pipe(texts.astype(str), batch_size=batch_size):
        results.append({ent.text: ent.label_ for ent in doc.ents})
    return results

# ---- RUN IN BATCHES (FAST) ----
df["target_hsent_spacy_ents"]  = extract_ents_spacy_batch(df[TARGET_H])
df["target_gsent_spacy_ents"]  = extract_ents_spacy_batch(df[TARGET_G])
df["matched_hsent_spacy_ents"] = extract_ents_spacy_batch(df[MATCHED_H])

# df.to_csv("sent_matches_102615_spacyEnts.csv", index=False)

print("spaCy entities extracted (BATCH MODE).")

df.head()


spaCy entities extracted (BATCH MODE).


,htid,target_hsent,target_gsent,gid,source_row_index,matched_hsent,source_df,csv_source,domain,pubdate,cidx,sidx,cer,wer,target_hsent_spacy_ents,target_gsent_spacy_ents,matched_hsent_spacy_ents
0,uc2.ark+=13960=t7dr2vk0r,"Besides , you would be miserable , managing na...","Besides , you would be miserable , managing na...",56600,10,"Besides , you would be miserable , managing \n...",ht_matches_4,highpairs,Fiction,1870,0,17,0.054348,0.138889,{},{},{}
1,uc2.ark+=13960=t7dr2vk0r,"Lewis will not feel it as we sbali , be bas b...",Lewis will not feel it as we shall he has bee...,56600,14,"— Lewis will not feel it as we sbali , — be ba...",ht_matches_4,highpairs,Fiction,1870,0,24,0.203125,0.500000,"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{'Lewis': 'PERSON'},"{'Lewis': 'PERSON', 'awav': 'PERSON'}"
2,uc2.ark+=13960=t7dr2vk0r,Frere wrote us word he was the taller of tiie ...,Frere wrote us word he was the taller of the t...,56600,17,""" Mr. Frere wrote us word he was the taller of...",ht_matches_4,highpairs,Fiction,1870,0,31,0.044643,0.115385,"{'Frere': 'PERSON', 'tiie': 'PERSON', 'two': '...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half':...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half a..."
3,uc2.ark+=13960=t7dr2vk0r,"Considerably above the middle height , his fig...","Considerably above the middle height , his fig...",56600,21,"Considerably above the \n middle height , his...",pg_matches,highpairs,Fiction,1870,0,40,0.045802,0.136364,{},{},{}
4,uc2.ark+=13960=t7dr2vk0r,"The only clue we have been able to gain is , t...",The only clue we have been able to gain is tha...,56600,36,The only clue we \n have been able to gain is...,pg_matches,highpairs,Fiction,1870,0,77,0.098039,0.250000,{},{},{}


In [ ]:
# df.to_csv("sent_matches_102615_spacyEnts.csv", index=False)

In [17]:
print(df['target_hsent_spacy_ents'].apply(len).sum())

# print(df['target_gsent_spacy_ents'].apply(lambda x: sum(int(k) for k in x.keys()) if x else 0))
print(df['target_gsent_spacy_ents'].apply(len).sum())

# print(df['matched_hsent_spacy_ents'].apply(lambda x: sum(int(k) for k in x.keys()) if x else 0))
print(df['matched_hsent_spacy_ents'].apply(len).sum())

102633
91747
98981


In [ ]:
# ner = nlp.get_pipe("ner")

# ner_labels = list(ner.labels)
# # print(ner_labels)
# # print(type(ner_labels))

# for label in ner_labels:
#     print(f"{label}: {spacy.explain(label)}")


In [ ]:
my_api_key = 'HIDDEN-NICE-TRY!'


NER with GPT-4o (requires input of your own OpenAI api key)

In [67]:
# NER with GPT-4o -- currently zero-shot
client = OpenAI(api_key=my_api_key)

df = pd.read_csv("sent_matches_102615_spacyGptEnts_first1k.csv")
print(df.shape)
# df = sent_df

TARGET_H = "target_hsent"
TARGET_G = "target_gsent"
MATCHED_H = "matched_hsent"

import re

def safe_json_extract(txt):
    match = re.search(r"\{.*\}", txt, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group())
    except:
        return {}

def extract_ents_gpt(sentence):
    if pd.isna(sentence):
        return {}
    
    prompt = f"""
You are a deterministic named entity recognition system.

TASK:
Extract named entities from the sentence below and return ONLY a valid Python dictionary
mapping entity text to entity label.

FORMAT:
Return ONLY a JSON object like this:
{{"Paris": "GPE", "Napoleon": "PERSON"}}

RULES:
- Use ONLY spaCy en_core_web_trf entity labels
- If no entities are present, return {{}} EXACTLY
- Do NOT include explanations
- Do NOT include markdown
- Do NOT include any text outside the JSON

ENTITY LABELS REFERENCE:
https://raw.githubusercontent.com/rdubnic2/htrc_misc/refs/heads/master/spacy_ner_en_core_web_trf_entities_labels.txt

SENTENCE:
{sentence}
"""
    
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    txt = resp.choices[0].message.content.strip()

    # if txt != '{}':
    #     # print("RAW GPT OUTPUT:", txt)

    try:
        return safe_json_extract(txt)
    except:
        return {}

tqdm.pandas()

df["target_hsent_gpt4o_ents"] = df[TARGET_H].head(1000).progress_apply(extract_ents_gpt)
target_hsent_ent_sum = df["target_hsent_gpt4o_ents"].dropna().apply(len).sum()
print(f"Found {target_hsent_ent_sum} entities within 'target_hsent' sentences.")

df["target_gsent_gpt4o_ents"] = df[TARGET_G].head(1000).progress_apply(extract_ents_gpt)
target_gsent_ent_sum = df["target_gsent_gpt4o_ents"].dropna().apply(len).sum()
print(f"Found {target_gsent_ent_sum} entities within 'target_gsent' sentences.")

# df["matched_hsent_gpt4o_ents"] = df[MATCHED_H].head(1000).progress_apply(extract_ents_gpt)
# matched_hsent_ent_sum = df["matched_hsent_gpt4o_ents"].dropna().apply(len).sum()
# print(f"Found {matched_hsent_ent_sum} entities within 'matched_hsent' sentences.")

df.to_csv("sent_matches_102615_spacyGptEnts_first1k_2.csv", index=False)

print(f"Total entities extracted using all classes of sentences: {target_hsent_ent_sum + target_gsent_ent_sum}")

df.head(10)

# big run started with 9.93 in openai account!
# FULL RUN WILL TAKE A LONG TIME!!

/var/folders/k9/cjwycg9j25503fvmwqtbz_dc0000gq/T/ipykernel_73824/649360848.py:4: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("sent_matches_102615_spacyGptEnts_first1k.csv")


(102615, 18)


100%|██████████| 1000/1000 [08:29<00:00,  1.96it/s]


Found 181 entities within 'target_hsent' sentences.


100%|██████████| 1000/1000 [08:26<00:00,  1.97it/s]


Found 235 entities within 'target_gsent' sentences.
Total entities extracted using all classes of sentences: 416


,htid,target_hsent,target_gsent,gid,source_row_index,matched_hsent,source_df,csv_source,domain,pubdate,cidx,sidx,cer,wer,target_hsent_spacy_ents,target_gsent_spacy_ents,matched_hsent_spacy_ents,matched_hsent_gpt4o_ents,target_hsent_gpt4o_ents,target_gsent_gpt4o_ents
0,uc2.ark+=13960=t7dr2vk0r,"Besides , you would be miserable , managing na...","Besides , you would be miserable , managing na...",56600,10,"Besides , you would be miserable , managing \n...",ht_matches_4,highpairs,Fiction,1870,0,17,0.054348,0.138889,{},{},{},{},{},{}
1,uc2.ark+=13960=t7dr2vk0r,"Lewis will not feel it as we sbali , be bas b...",Lewis will not feel it as we shall he has bee...,56600,14,"— Lewis will not feel it as we sbali , — be ba...",ht_matches_4,highpairs,Fiction,1870,0,24,0.203125,0.500000,"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{'Lewis': 'PERSON'},"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{},{},{}
2,uc2.ark+=13960=t7dr2vk0r,Frere wrote us word he was the taller of tiie ...,Frere wrote us word he was the taller of the t...,56600,17,""" Mr. Frere wrote us word he was the taller of...",ht_matches_4,highpairs,Fiction,1870,0,31,0.044643,0.115385,"{'Frere': 'PERSON', 'tiie': 'PERSON', 'two': '...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half':...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half a...",{},{},{}
3,uc2.ark+=13960=t7dr2vk0r,"Considerably above the middle height , his fig...","Considerably above the middle height , his fig...",56600,21,"Considerably above the \n middle height , his...",pg_matches,highpairs,Fiction,1870,0,40,0.045802,0.136364,{},{},{},{},{},{}
4,uc2.ark+=13960=t7dr2vk0r,"The only clue we have been able to gain is , t...",The only clue we have been able to gain is tha...,56600,36,The only clue we \n have been able to gain is...,pg_matches,highpairs,Fiction,1870,0,77,0.098039,0.250000,{},{},{},{},{},{}
5,uc2.ark+=13960=t7dr2vk0r,"nor woiddn't , if you was to double my wages ,...","nor would n't , if you was to double my wages ...",56600,54,"nor woiddn't , \n if you was to double my wag...",ht_matches_4,highpairs,Fiction,1870,0,114,0.062500,0.120000,{},{},{},{},{},{}
6,uc2.ark+=13960=t7dr2vk0r,"During the evening , in the cotn se of convei...","During the evening , in the course of conversa...",56600,64,"During the evening , in the cotn - se of conve...",pg_matches,highpairs,Fiction,1870,0,139,0.070866,0.217391,{'Arundel': 'PERSON'},{'Arundel': 'PERSON'},{'Arundel': 'PERSON'},{},{'Mrs. Arundel': 'PERSON'},{'Mrs. Arundel': 'PERSON'}
7,uc2.ark+=13960=t7dr2vk0r,"He is as largo as a calf , which is inconvenie...","He is as large as a calf , which is inconvenie...",56600,87,"He is as largo as a calf , which is \n inconv...",pg_matches,highpairs,Fiction,1870,0,193,0.042553,0.136364,{},{},{},{},{},{}
8,uc2.ark+=13960=t7dr2vk0r,I ca n't live among stmngere without something...,I ca n't live among strangers without somethin...,56600,89,I can't live among stmngere without some- \n ...,pg_matches,highpairs,Fiction,1870,0,195,0.053030,0.133333,{'stmngere': 'ORG'},{},{'stmngere': 'NORP'},{},{},{}
9,uc2.ark+=13960=t7dr2vk0r,Ca n't you write to me to morrow T H,"Ca n't you write to me to morrow ? """,56600,90,Ca n't you write to me to - morrow T \n \n,ht_matches_4,highpairs,Fiction,1870,0,196,0.083333,0.200000,{'morrow': 'DATE'},{'morrow': 'DATE'},{},{},{},{}


In [63]:
# df.to_csv("sent_matches_102615_spacyGptEnts_oldhsent20233rows.csv", index=False)
df.head(20)

,htid,target_hsent,target_gsent,gid,source_row_index,matched_hsent,source_df,csv_source,domain,pubdate,cidx,sidx,cer,wer,target_hsent_spacy_ents,target_gsent_spacy_ents,matched_hsent_spacy_ents,matched_hsent_gpt4o_ents
0,uc2.ark+=13960=t7dr2vk0r,"Besides , you would be miserable , managing na...","Besides , you would be miserable , managing na...",56600,10,"Besides , you would be miserable , managing \n...",ht_matches_4,highpairs,Fiction,1870,0,17,0.054348,0.138889,{},{},{},{}
1,uc2.ark+=13960=t7dr2vk0r,"Lewis will not feel it as we sbali , be bas b...",Lewis will not feel it as we shall he has bee...,56600,14,"— Lewis will not feel it as we sbali , — be ba...",ht_matches_4,highpairs,Fiction,1870,0,24,0.203125,0.500000,"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{'Lewis': 'PERSON'},"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{}
2,uc2.ark+=13960=t7dr2vk0r,Frere wrote us word he was the taller of tiie ...,Frere wrote us word he was the taller of the t...,56600,17,""" Mr. Frere wrote us word he was the taller of...",ht_matches_4,highpairs,Fiction,1870,0,31,0.044643,0.115385,"{'Frere': 'PERSON', 'tiie': 'PERSON', 'two': '...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half':...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half a...",{}
3,uc2.ark+=13960=t7dr2vk0r,"Considerably above the middle height , his fig...","Considerably above the middle height , his fig...",56600,21,"Considerably above the \n middle height , his...",pg_matches,highpairs,Fiction,1870,0,40,0.045802,0.136364,{},{},{},{}
4,uc2.ark+=13960=t7dr2vk0r,"The only clue we have been able to gain is , t...",The only clue we have been able to gain is tha...,56600,36,The only clue we \n have been able to gain is...,pg_matches,highpairs,Fiction,1870,0,77,0.098039,0.250000,{},{},{},{}
5,uc2.ark+=13960=t7dr2vk0r,"nor woiddn't , if you was to double my wages ,...","nor would n't , if you was to double my wages ...",56600,54,"nor woiddn't , \n if you was to double my wag...",ht_matches_4,highpairs,Fiction,1870,0,114,0.062500,0.120000,{},{},{},{}
6,uc2.ark+=13960=t7dr2vk0r,"During the evening , in the cotn se of convei...","During the evening , in the course of conversa...",56600,64,"During the evening , in the cotn - se of conve...",pg_matches,highpairs,Fiction,1870,0,139,0.070866,0.217391,{'Arundel': 'PERSON'},{'Arundel': 'PERSON'},{'Arundel': 'PERSON'},{}
7,uc2.ark+=13960=t7dr2vk0r,"He is as largo as a calf , which is inconvenie...","He is as large as a calf , which is inconvenie...",56600,87,"He is as largo as a calf , which is \n inconv...",pg_matches,highpairs,Fiction,1870,0,193,0.042553,0.136364,{},{},{},{}
8,uc2.ark+=13960=t7dr2vk0r,I ca n't live among stmngere without something...,I ca n't live among strangers without somethin...,56600,89,I can't live among stmngere without some- \n ...,pg_matches,highpairs,Fiction,1870,0,195,0.053030,0.133333,{'stmngere': 'ORG'},{},{'stmngere': 'NORP'},{}
9,uc2.ark+=13960=t7dr2vk0r,Ca n't you write to me to morrow T H,"Ca n't you write to me to morrow ? """,56600,90,Ca n't you write to me to - morrow T \n \n,ht_matches_4,highpairs,Fiction,1870,0,196,0.083333,0.200000,{'morrow': 'DATE'},{'morrow': 'DATE'},{},{}


In [ ]:
# NER with GPT-4o -- currently 1-shot
client = OpenAI(api_key=my_api_key)

df = pd.read_csv('sent_matches_102615_spacyEnts.csv')
print(df.shape)
# df = sent_df

TARGET_H = "target_hsent"
TARGET_G = "target_gsent"
MATCHED_H = "matched_hsent"

import re

def safe_json_extract(txt):
    match = re.search(r"\{.*\}", txt, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group())
    except:
        return {}

def extract_ents_gpt(sentence):
    if pd.isna(sentence):
        return {}
    
    prompt = f"""
You are a deterministic named entity recognition system.

TASK:
Extract named entities from the sentence below and return ONLY a valid Python dictionary
mapping entity text to entity label.

FORMAT:
Return ONLY a JSON object like this:
{{"Paris": "GPE", "Napoleon": "PERSON"}}

RULES:
- Use ONLY spaCy en_core_web_trf entity labels
- If no entities are present, return {{}} EXACTLY
- Do NOT include explanations
- Do NOT include markdown
- Do NOT include any text outside the JSON

ENTITY LABELS REFERENCE:
https://raw.githubusercontent.com/rdubnic2/htrc_misc/refs/heads/master/spacy_ner_en_core_web_trf_entities_labels.txt

SENTENCE:
{sentence}
"""
    
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    txt = resp.choices[0].message.content.strip()

    if txt != '{}':
        # print("RAW GPT OUTPUT:", txt)

    try:
        return safe_json_extract(txt)
    except:
        return {}

tqdm.pandas()

df["target_hsent_gpt4o_ents"] = df[TARGET_H].progress_apply(extract_ents_gpt)
target_hsent_ent_sum = df["target_hsent_gpt4o_ents"].dropna().apply(len).sum()
print(f"Found {target_hsent_ent_sum} entities within 'target_hsent' sentences.")

df["target_gsent_gpt4o_ents"] = df[TARGET_G].progress_apply(extract_ents_gpt)
target_gsent_ent_sum = df["target_gsent_gpt4o_ents"].dropna().apply(len).sum()
print(f"Found {target_gsent_ent_sum} entities within 'target_gsent' sentences.")

df["matched_hsent_gpt4o_ents"] = df[MATCHED_H].progress_apply(extract_ents_gpt)
matched_hsent_ent_sum = df["matched_hsent_gpt4o_ents"].dropna().apply(len).sum()
print(f"Found {matched_hsent_ent_sum} entities within 'matched_hsent' sentences.")

# df.to_csv("sent_matches_102615_spacyGptEnts.csv", index=False)

print(f"Total entities extracted using all classes of sentences: {target_hsent_ent_sum + target_gsent_ent_sum + matched_hsent_ent_sum}")

df.head(10)

# big run started with 9.93 in openai account!
# FULL RUN WILL TAKE A LONG TIME!!

In [ ]:
# df = df.drop(['target_gsent_ents','matched_hsent_ents'], axis=1)
# df.head()

,htid,target_hsent,target_gsent,gid,source_row_index,matched_hsent,source_df,csv_source,domain,pubdate,cidx,sidx,cer,wer,target_hsent_spacy_ents,target_gsent_spacy_ents,matched_hsent_spacy_ents,target_hsent_gpt4o_ents
0,uc2.ark+=13960=t7dr2vk0r,"Besides , you would be miserable , managing na...","Besides , you would be miserable , managing na...",56600,10,"Besides , you would be miserable , managing \n...",ht_matches_4,highpairs,Fiction,1870,0,17,0.054348,0.138889,{},{},{},{}
1,uc2.ark+=13960=t7dr2vk0r,"Lewis will not feel it as we sbali , be bas b...",Lewis will not feel it as we shall he has bee...,56600,14,"— Lewis will not feel it as we sbali , — be ba...",ht_matches_4,highpairs,Fiction,1870,0,24,0.203125,0.500000,"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{'Lewis': 'PERSON'},"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{}
2,uc2.ark+=13960=t7dr2vk0r,Frere wrote us word he was the taller of tiie ...,Frere wrote us word he was the taller of the t...,56600,17,""" Mr. Frere wrote us word he was the taller of...",ht_matches_4,highpairs,Fiction,1870,0,31,0.044643,0.115385,"{'Frere': 'PERSON', 'tiie': 'PERSON', 'two': '...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half':...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half a...",{}
3,uc2.ark+=13960=t7dr2vk0r,"Considerably above the middle height , his fig...","Considerably above the middle height , his fig...",56600,21,"Considerably above the \n middle height , his...",pg_matches,highpairs,Fiction,1870,0,40,0.045802,0.136364,{},{},{},{}
4,uc2.ark+=13960=t7dr2vk0r,"The only clue we have been able to gain is , t...",The only clue we have been able to gain is tha...,56600,36,The only clue we \n have been able to gain is...,pg_matches,highpairs,Fiction,1870,0,77,0.098039,0.250000,{},{},{},{}


In [21]:
for sent in df.target_hsent.head(10).values:
    print(sent)
    print()

Besides , you would be miserable , managing naughty children qll day long  throwing away your talents on a set of stupid little wretches ,  such di  udgery would ennui you to death . "

Lewis will not feel it as we sbali ,  be bas been awav 60 Ions ; . "

Frere wrote us word he was the taller of tiie two by half a head last year , if you recollect , " retiu  ned Rose .

Considerably above the middle height , his figure was slender , but singularly gi  aceful ; his head small , and intellectual  looking .

The only clue we have been able to gain is , that ^Ir .

nor woiddn't , if you was to double my wages , and put the washin out  I ca n't abear them . "

During the evening , in the cotn  se of convei'sation , Mrs. Arundel again refen  ed to the project of teaching music and singing .

He is as largo as a calf , which is inconvenient , and I doubt whether he is full  gi  own yet .

I ca n't live among stmngere without something to love , and that loves me ; so dotv't wony me about it ,

Below code is incomplete given the roadblocks to continuing the replication

In [ ]:
# NER with mistral-large-2512
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage
import time

api_key = 'MIYhiZ1UNb8Ah6h9eKN0PoHcESaZanBj'
mistral_model = 

client = MistralClient()

df = pd.read_csv("sent_matches_deduped_final_1024.csv")

TARGET_H = "target_hsent"
TARGET_G = "target_gsent"
MATCHED_H = "matched_hsent"

# ---- PROMPT TEMPLATE ----
MISTRAL_NER_PROMPT = """
You are an expert named-entity recognition system.

Extract all named entities from the sentence below and return
ONLY a valid JSON dictionary in this exact format:

{"entity": "ENTITY_TYPE"}

Use ONLY these entity types:
PERSON, ORG, GPE, LOC, FACILITY, WORK_OF_ART, EVENT, DATE, PRODUCT, LAW, LANGUAGE, NORP, OTHER

If there are no named entities, return {}.

Sentence:
"""

# ner function
def extract_ents_mistral(sentence):
    if pd.isna(sentence) or not str(sentence).strip():
        return {}
    
    prompt = MISTRAL_NER_PROMPT + str(sentence)

    try:
        messages = [ChatMessage(role="user", content=prompt)]

        response = client.chat(
            model="mistral-large-latest",
            messages=messages,
            temperature=0.0
        )

        txt = response.choices[0].message.content.strip()

        # --- Strict JSON recovery ---
        start = txt.find("{")
        end = txt.rfind("}") + 1
        txt = txt[start:end]

        return json.loads(txt)

    except Exception as e:
        return {"__error__": str(e)}

tqdm.pandas()

# ---- RUN MISTRAL NER ----
df["target_hsent_ents_mistral"] = df[TARGET_H].progress_apply(extract_ents_mistral)
df["target_gsent_ents_mistral"] = df[TARGET_G].progress_apply(extract_ents_mistral)
df["matched_hsent_ents_mistral"] = df[MATCHED_H].progress_apply(extract_ents_mistral)

# ---- SAVE ----
df.to_csv("sent_matches_with_mistral_entities.csv", index=False)

print("Mistral NER extraction complete.")


Compare outputs, maybe through: 
* entity types captured
* raw entites captured? 

Will be hard to figure out how to evaluate without manual review.

#### Reviewing NER results

In [73]:
df = pd.read_csv("sent_matches_102615_spacyGptEnts_first1k_2.csv")
df.head(25)

/var/folders/k9/cjwycg9j25503fvmwqtbz_dc0000gq/T/ipykernel_73824/1162163356.py:1: DtypeWarning: Columns (17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("sent_matches_102615_spacyGptEnts_first1k_2.csv")


,htid,target_hsent,target_gsent,gid,source_row_index,matched_hsent,source_df,csv_source,domain,pubdate,cidx,sidx,cer,wer,target_hsent_spacy_ents,target_gsent_spacy_ents,matched_hsent_spacy_ents,matched_hsent_gpt4o_ents,target_hsent_gpt4o_ents,target_gsent_gpt4o_ents
0,uc2.ark+=13960=t7dr2vk0r,"Besides , you would be miserable , managing na...","Besides , you would be miserable , managing na...",56600,10,"Besides , you would be miserable , managing \n...",ht_matches_4,highpairs,Fiction,1870,0,17,0.054348,0.138889,{},{},{},{},{},{}
1,uc2.ark+=13960=t7dr2vk0r,"Lewis will not feel it as we sbali , be bas b...",Lewis will not feel it as we shall he has bee...,56600,14,"— Lewis will not feel it as we sbali , — be ba...",ht_matches_4,highpairs,Fiction,1870,0,24,0.203125,0.500000,"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{'Lewis': 'PERSON'},"{'Lewis': 'PERSON', 'awav': 'PERSON'}",{},{},{}
2,uc2.ark+=13960=t7dr2vk0r,Frere wrote us word he was the taller of tiie ...,Frere wrote us word he was the taller of the t...,56600,17,""" Mr. Frere wrote us word he was the taller of...",ht_matches_4,highpairs,Fiction,1870,0,31,0.044643,0.115385,"{'Frere': 'PERSON', 'tiie': 'PERSON', 'two': '...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half':...","{'Frere': 'PERSON', 'two': 'CARDINAL', 'half a...",{},{},{}
3,uc2.ark+=13960=t7dr2vk0r,"Considerably above the middle height , his fig...","Considerably above the middle height , his fig...",56600,21,"Considerably above the \n middle height , his...",pg_matches,highpairs,Fiction,1870,0,40,0.045802,0.136364,{},{},{},{},{},{}
4,uc2.ark+=13960=t7dr2vk0r,"The only clue we have been able to gain is , t...",The only clue we have been able to gain is tha...,56600,36,The only clue we \n have been able to gain is...,pg_matches,highpairs,Fiction,1870,0,77,0.098039,0.250000,{},{},{},{},{},{}
5,uc2.ark+=13960=t7dr2vk0r,"nor woiddn't , if you was to double my wages ,...","nor would n't , if you was to double my wages ...",56600,54,"nor woiddn't , \n if you was to double my wag...",ht_matches_4,highpairs,Fiction,1870,0,114,0.062500,0.120000,{},{},{},{},{},{}
6,uc2.ark+=13960=t7dr2vk0r,"During the evening , in the cotn se of convei...","During the evening , in the course of conversa...",56600,64,"During the evening , in the cotn - se of conve...",pg_matches,highpairs,Fiction,1870,0,139,0.070866,0.217391,{'Arundel': 'PERSON'},{'Arundel': 'PERSON'},{'Arundel': 'PERSON'},{},{'Mrs. Arundel': 'PERSON'},{'Mrs. Arundel': 'PERSON'}
7,uc2.ark+=13960=t7dr2vk0r,"He is as largo as a calf , which is inconvenie...","He is as large as a calf , which is inconvenie...",56600,87,"He is as largo as a calf , which is \n inconv...",pg_matches,highpairs,Fiction,1870,0,193,0.042553,0.136364,{},{},{},{},{},{}
8,uc2.ark+=13960=t7dr2vk0r,I ca n't live among stmngere without something...,I ca n't live among strangers without somethin...,56600,89,I can't live among stmngere without some- \n ...,pg_matches,highpairs,Fiction,1870,0,195,0.053030,0.133333,{'stmngere': 'ORG'},{},{'stmngere': 'NORP'},{},{},{}
9,uc2.ark+=13960=t7dr2vk0r,Ca n't you write to me to morrow T H,"Ca n't you write to me to morrow ? """,56600,90,Ca n't you write to me to - morrow T \n \n,ht_matches_4,highpairs,Fiction,1870,0,196,0.083333,0.200000,{'morrow': 'DATE'},{'morrow': 'DATE'},{},{},{},{}


In [65]:
rev_df = pd.DataFrame()

In [76]:
# evaluating entity overlap
spacy_hsent_col = "target_hsent_spacy_ents"
spacy_gsent_col = "target_gsent_spacy_ents"
spacy_matched_hsent_col = "matched_hsent_spacy_ents"
gpt_hsent_col  = "target_hsent_gpt4o_ents"
gpt_gsent_col = "matched_gsent_gpt4o_ents"
gpt_matched_hsent_col = "matched_hsent_gpt4o_ents"


def entity_overlap(row, a, b):
    if not isinstance(row[a], dict) or not isinstance(row[b], dict):
        return 0
    return len(set(row[a].keys()) & set(row[b].keys()))

rev_df["gpt_spacy_target_hsent_overlap"] = df.head(1000).apply(lambda r: entity_overlap(r, spacy_hsent_col, gpt_hsent_col), axis=1)
rev_df["gpt_spacy_target_gsent_overlap"] = df.head(1000).apply(lambda r: entity_overlap(r, spacy_gsent_col, gpt_gsent_col), axis=1)
rev_df["gpt_spacy_matched_hsent_overlap"] = df.head(1000).apply(lambda r: entity_overlap(r, spacy_matched_hsent_col, gpt_matched_hsent_col), axis=1)

rev_df.head()
# rev_df["gpt_llama_overlap"] = df.apply(lambda r: entity_overlap(r, gpt_col, llama_col), axis=1)


,gpt_spacy_matched_hsent_overlap,gpt_spacy_target_hsent_overlap,gpt_spacy_target_gsent_overlap
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,0.0


In [77]:
rev_df.gpt_spacy_matched_hsent_overlap.value_counts()

gpt_spacy_matched_hsent_overlap
0.0    1000
Name: count, dtype: int64